In [1]:
import numpy as np
import pandas as pd

# Варіант 6
# Місто: Чернігів
# Базова ціна: 690 грн
# Базова кількість: 4

orders = pd.DataFrame([
    {"id": 1, "місто": "Чернігів",      "ціна": "690 грн", "кількість": 4},
    {"id": 2, "місто": " Чернігів ",    "ціна": "690 грн", "кількість": 3},
    {"id": 3, "місто": "ЧЕРНІГІВ",      "ціна": 690,       "кількість": np.nan},
    {"id": 4, "місто": "чернігів",      "ціна": 690,       "кількість": 5},
    {"id": 5, "місто": "Чернігів",      "ціна": 690,       "кількість": np.nan},
    {"id": 6, "місто": " Чернігів",     "ціна": 690,       "кількість": 4},
    {"id": 7, "місто": "ЧЕРНІГІВ ",     "ціна": 6900,      "кількість": 2},
    {"id": 7, "місто": "ЧЕРНІГІВ ",     "ціна": 6900,      "кількість": 2},  # дублікат
])

orders


,id,місто,ціна,кількість
0,1,Чернігів,690 грн,4.0
1,2,Чернігів,690 грн,3.0
2,3,ЧЕРНІГІВ,690,NaN
3,4,чернігів,690,5.0
4,5,Чернігів,690,NaN
5,6,Чернігів,690,4.0
6,7,ЧЕРНІГІВ,6900,2.0
7,7,ЧЕРНІГІВ,6900,2.0


In [2]:
# Кількість пропусків у кожному стовпці
orders.isna().sum()


id           0
місто        0
ціна         0
кількість    2
dtype: int64

In [3]:
median_quantity = orders["кількість"].median()

orders["кількість"] = orders["кількість"].fillna(median_quantity)

print("Медіана кількості:", median_quantity)
orders


Медіана кількості: 3.5


,id,місто,ціна,кількість
0,1,Чернігів,690 грн,4.0
1,2,Чернігів,690 грн,3.0
2,3,ЧЕРНІГІВ,690,3.5
3,4,чернігів,690,5.0
4,5,Чернігів,690,3.5
5,6,Чернігів,690,4.0
6,7,ЧЕРНІГІВ,6900,2.0
7,7,ЧЕРНІГІВ,6900,2.0


In [4]:
# Завдання 3 
# # Дублікати без subset
orders.duplicated().sum()


np.int64(1)

In [5]:
orders[orders.duplicated()]


,id,місто,ціна,кількість
7,7,ЧЕРНІГІВ,6900,2.0


In [6]:
orders.duplicated(subset=["id"]).sum()


np.int64(1)

In [7]:
orders = orders.drop_duplicates(subset=["id"])

print("Кількість рядків після видалення дубліката:", len(orders))


Кількість рядків після видалення дубліката: 7


In [8]:
# завдання 4
orders.dtypes


id             int64
місто            str
ціна          object
кількість    float64
dtype: object

In [9]:
orders["ціна"] = (
    orders["ціна"]
    .astype(str)
    .str.replace(" грн", "", regex=False)
    .astype(float)
)

orders.dtypes


id             int64
місто            str
ціна         float64
кількість    float64
dtype: object

In [10]:
orders["місто"] = orders["місто"].str.strip().str.lower()

orders["місто"].unique()


<StringArray>
['чернігів']
Length: 1, dtype: str

In [11]:
# завдання 5
q1 = orders["ціна"].quantile(0.25)
q3 = orders["ціна"].quantile(0.75)

iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print("Q1 =", q1)
print("Q3 =", q3)
print("IQR =", iqr)
print("Нижня межа =", lower)
print("Верхня межа =", upper)


Q1 = 690.0
Q3 = 690.0
IQR = 0.0
Нижня межа = 690.0
Верхня межа = 690.0


In [12]:
outliers = orders[
    (orders["ціна"] < lower) |
    (orders["ціна"] > upper)
]

outliers


,id,місто,ціна,кількість
6,7,чернігів,6900.0,2.0


In [13]:
#Письмове пояснення завдання 5
Значення 6900 грн я б вважав помилкою вводу, а не реальним великим замовленням. Базова ціна товару становить 690 грн, тому 6900 грн рівно у 10 разів більша. У цьому наборі немає інших ознак того, що товар міг коштувати настільки дорого. Найімовірніше, під час введення ціни було випадково додано зайвий нуль. Тому це значення доцільно виправити на 690 грн або видалити після перевірки первинного джерела даних.

SyntaxError: invalid syntax (2739346741.py, line 2)

In [14]:
#Три контрольні питання
#1. Чому середнє замість медіани може спотворити стандартне відхилення?
#Середнє значення сильно залежить від крайніх спостережень. Якщо в наборі є велике або дуже мале значення, воно може змістити середнє. Після заповнення пропусків цим зміщеним значенням частина даних штучно наближається до середнього. У результаті варіативність набору може бути недооцінена, а стандартне відхилення — спотворене. Медіана є стійкішою до викидів, тому в малому наборі вона часто є безпечнішим способом заповнення пропусків.

#2. Різниця між duplicated() без subset і з subset
#duplicated() без subset порівнює всі стовпці рядка. Рядки вважаються дублікати лише тоді, коли повністю збігаються. Якщо використовувати subset, наприклад duplicated(subset=["id"]), перевіряються тільки вибрані стовпці. Тому два рядки можуть бути дублікати за id, навіть якщо інші їхні значення відрізняються.

#3. Чому не можна автоматично видаляти всі значення поза межами IQR?
#Значення поза межами IQR не обов'язково є помилками. Вони можуть бути рідкісними, але цілком реальними спостереженнями. Наприклад, велике замовлення справді може мати значно вищу ціну або кількість товару. Тому IQR потрібно використовувати як спосіб виявлення підозрілих значень, а остаточне рішення про видалення треба приймати з урахуванням предметної області та походження даних.